EXPLORE DATA

CHARGEMENT

In [15]:
import duckdb
import urllib.request
import zipfile
import os

# ── 1. Téléchargement et extraction du fichier ──────────────────────────────
url = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20260405-002321/valeursfoncieres-2025.txt.zip"
zip_path = "valeursfoncieres-2025.zip"
txt_path = "valeursfoncieres-2025.txt"

print("Téléchargement en cours...")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(".")
    txt_path = z.namelist()[0]  # récupère le nom exact du fichier extrait

print(f"Fichier extrait : {txt_path}")

# ── 2. Connexion DuckDB et chargement ────────────────────────────────────────
con = duckdb.connect("")

# Le fichier DVF est séparé par des "|" (pipe)
con.execute(f"""
    CREATE TABLE dvf AS
    SELECT * FROM read_csv_auto('{txt_path}',
        delim='|',
        header=True,
        ignore_errors=True
    )
""")

# ── 3. Nombre total de lignes ─────────────────────────────────────────────────
total = con.execute("SELECT COUNT(*) FROM dvf").fetchone()[0]
print(f"\n Nombre total de lignes : {total:,}")

# ── 4. Variables à conserver ─────────────────────────────────────────────────

cols = [row[0] for row in con.execute("DESCRIBE dvf").fetchall()]
print(f"\n Toutes les colonnes disponibles ({len(cols)}) :")
print(cols)

# Colonnes cibles (avec wildcard adresse_*)
cols_to_keep_explicit = [
    "Valeur fonciere", "Surface reelle bati",
    "Nombre pieces principales", "Type local","Nature mutation",
    "Code commune", "Date mutation","Code postal","Commune","Code departement","Code voie", "Voie"
]
cols_to_keep = cols_to_keep_explicit 

# Garder uniquement les colonnes qui existent réellement
cols_to_keep = [c for c in cols_to_keep if c in cols]
print(f"\n Colonnes conservées ({len(cols_to_keep)}) : {cols_to_keep}")

# ── 5. Valeurs manquantes par colonne ────────────────────────────────────────
print("\n Valeurs manquantes par colonne :")
for col in cols_to_keep:
    nulls = con.execute(f"SELECT COUNT(*) FROM dvf WHERE \"{col}\" IS NULL").fetchone()[0]
    pct = round(100 * nulls / total, 2)
    print(f"  {col:<35} : {nulls:>10,} manquantes  ({pct}%)")

# ── 6. Création de la table nettoyée (colonnes sélectionnées) ────────────────
select_clause = ", ".join([f'"{c}"' for c in cols_to_keep])
con.execute(f"""
    CREATE TABLE dvf_clean AS
    SELECT {select_clause}
    FROM dvf
""")

clean_count = con.execute("SELECT COUNT(*) FROM dvf_clean").fetchone()[0]
print(f"\n Table dvf_clean créée avec {clean_count:,} lignes et {len(cols_to_keep)} colonnes")

# ── 7. Nettoyage des valeurs manquantes ──────────────────────────────────────

# 1. Supprimer les lignes où valeur_fonciere est NULL
con.execute("""
    CREATE TABLE dvf_clean2 AS
    SELECT * FROM dvf_clean
    WHERE "Valeur fonciere" IS NOT NULL
""")

count_after = con.execute("SELECT COUNT(*) FROM dvf_clean2").fetchone()[0]
print(f" Lignes après suppression des NULL sur Valeur fonciere : {count_after:,}")

# 2. Remplacer les NULL par 0 pour les colonnes numériques
cols_numeric = [
    "Surface reelle bati",
    "Nombre pieces principales","Code postal"
]
for col in cols_numeric:
    con.execute(f"""
        UPDATE dvf_clean2
        SET "{col}" = 0
        WHERE "{col}" IS NULL
    """)
    print(f"  ✔ NULL remplacés par 0 dans : {col}")

# 3. Remplacer les NULL par 'NR' pour les colonnes texte
cols_text = [
    "Type local",
    "Code voie",
    "Voie"
]
for col in cols_text:
    con.execute(f"""
        UPDATE dvf_clean2
        SET "{col}" = 'NR'
        WHERE "{col}" IS NULL
    """)
    print(f"  ✔ NULL remplacés par 'NR' dans : {col}")

# ── 8. Vérification finale ───────────────────────────────────────────────────
print("\n Vérification après nettoyage :")
total_clean = con.execute("SELECT COUNT(*) FROM dvf_clean2").fetchone()[0]
print(f"  Nombre de lignes finales : {total_clean:,}")

for col in cols_numeric + cols_text + ["Valeur fonciere"]:
    nulls = con.execute(f'SELECT COUNT(*) FROM dvf_clean2 WHERE "{col}" IS NULL').fetchone()[0]
    print(f"  {col:<35} : {nulls} NULL restants")

# ── 9. Aperçu ────────────────────────────────────────────────────────────────
print("\n Aperçu des 5 premières lignes :")
print(con.execute("SELECT * FROM dvf_clean2 LIMIT 20").df().to_string())

Téléchargement en cours...
Fichier extrait : ValeursFoncieres-2025.txt


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Nombre total de lignes : 3,714,493

 Toutes les colonnes disponibles (43) :
['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']

 Colonnes conservées (12) : ['Valeur fonciere', 'Surface reelle bati', 'Nombre pieces principales', 'Type local', 'Nature mutation', 'Co

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Table dvf_clean créée avec 3,714,493 lignes et 12 colonnes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Lignes après suppression des NULL sur Valeur fonciere : 3,666,073
  ✔ NULL remplacés par 0 dans : Surface reelle bati
  ✔ NULL remplacés par 0 dans : Nombre pieces principales
  ✔ NULL remplacés par 0 dans : Code postal
  ✔ NULL remplacés par 'NR' dans : Type local
  ✔ NULL remplacés par 'NR' dans : Code voie
  ✔ NULL remplacés par 'NR' dans : Voie

 Vérification après nettoyage :
  Nombre de lignes finales : 3,666,073
  Surface reelle bati                 : 0 NULL restants
  Nombre pieces principales           : 0 NULL restants
  Code postal                         : 0 NULL restants
  Type local                          : 0 NULL restants
  Code voie                           : 0 NULL restants
  Voie                                : 0 NULL restants
  Valeur fonciere                     : 0 NULL restants

 Aperçu des 5 premières lignes :
   Valeur fonciere  Surface reelle bati  Nombre pieces principales   Type local Nature mutation  Code commune Date mutation  Code postal    Commune Co

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
DPE_FILE   = r"h:\Mes documents\sae601 29.05\sae_601\data\dpe.csv"
OUTPUT_DIR = r"h:\Mes documents\sae601 29.05\sae_601\data\\"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ETIQUETTES_VALIDES = ('A', 'B', 'C', 'D', 'E', 'F', 'G')

# ── Connexion DuckDB en mémoire ────────────────────────────────────────────────
con = duckdb.connect("dpe.db")
print("DuckDB connecté.")


# ------------------------------------------------------------------------------
# ── Suppression des tables existantes (idempotent) ────────────────────────────
# ------------------------------------------------------------------------------

for table in ["dpe_raw", "dpe_imputed", "dpe_clean", "dpe_dedup", "dpe_final"]:
    con.execute(f"DROP TABLE IF EXISTS {table}")
    print(f"  Table '{table}' supprimée (si existait)")

# ------------------------------------------------------------------------------
# ── 1. Chargement brut du CSV (colonnes ciblées uniquement) ───────────────────
# ------------------------------------------------------------------------------


COLS_TO_KEEP = [
    # Identification 
    "numero_dpe", "date_etablissement_dpe", "date_fin_validite_dpe",
    "date_derniere_modification_dpe",
    # Géolocalisation BAN
    "code_insee_ban", "code_departement_ban", "code_region_ban",
    "coordonnee_cartographique_x_ban", "coordonnee_cartographique_y_ban", "score_ban",
    "nom_commune_ban", "code_postal_ban","adresse_brut","nom_commune_brut","code_postal_brut",
    # Bilan DPE — variables cibles
    "etiquette_dpe", "etiquette_ges",
    "conso_5_usages_ep", "conso_5_usages_par_m2_ep",
    "conso_5 usages_ef", "conso_5 usages_par_m2_ef",
    "emission_ges_5_usages", "emission_ges_5_usages par_m2",
    # Coûts annuels
    "cout_total_5_usages", "cout_chauffage", "cout_ecs",
    "cout_refroidissement", "cout_eclairage", "cout_auxiliaires",
    # Caractéristiques logement — variables de contrôle
    "type_batiment", "typologie_logement", "surface_habitable_logement", "periode_construction","indicateur_confort_ete",
    "nombre_niveau_logement", "zone_climatique", "classe_altitude",
    # Chauffage & isolation
    "type_energie_principale_chauffage", "type_generateur_chauffage_principal",
    "qualite_isolation_enveloppe", "qualite_isolation_murs",
    "qualite_isolation_menuiseries", "ubat_w_par_m2_k", "isolation_toiture","conso_chauffage_ef"
]

# On vérifie d'abord quelles colonnes existent réellement dans le fichier
cols_header = con.execute(f"""
    SELECT column_name FROM (
        DESCRIBE SELECT * FROM read_csv('{DPE_FILE}', auto_detect=TRUE, all_varchar=TRUE, sample_size=1)
    )
""").fetchdf()["column_name"].tolist()

cols_ok      = [c for c in COLS_TO_KEEP if c in cols_header]
cols_absentes = [c for c in COLS_TO_KEEP if c not in cols_header]

print(f"  Colonnes conservées : {len(cols_ok)} / {len(COLS_TO_KEEP)}")
if cols_absentes:
    print(" Colonnes absentes :", cols_absentes)

# Chargement avec sélection immédiate — DuckDB ne lit que ces colonnes
select_clause = ", ".join([f'"{c}"' for c in cols_ok])

con.execute(f"""
    CREATE TABLE dpe_raw AS
    SELECT {select_clause}
    FROM read_csv(
        '{DPE_FILE}',
        auto_detect   = TRUE,
        all_varchar   = TRUE,
        ignore_errors = TRUE
    )
""")

total = con.execute("SELECT COUNT(*) FROM dpe_raw").fetchone()[0]
print(f"\n  Lignes chargées : {total:,}")
print(f"  Colonnes en mémoire : {len(cols_ok)}")


# ------------------------------------------------------------------------------
# ── 2. Valeurs manquantes (sur toutes les colonnes chargées) ──────────────────
# ------------------------------------------------------------------------------

total = con.execute("SELECT COUNT(*) FROM dpe_raw").fetchone()[0]

print(f"  Lignes totales : {total:,}\n")
print(f"  {'Colonne':<45} {'NULL':>10}  {'%':>6}")
print("  " + "-" * 65)

for col in cols_ok:
    n = con.execute(
        f'SELECT COUNT(*) FROM dpe_raw WHERE "{col}" IS NULL OR TRIM("{col}") = \'\''
    ).fetchone()[0]
    pct = round(100 * n / total, 2)
    flag = "  ⚠️" if pct > 20 else ("  ℹ️" if pct > 5 else "")
    print(f"  {col:<45} {n:>10,}  {pct:>5.1f} %{flag}")



# ── Remplacement des valeurs manquantes ───────────────────────────────────────

NUMERIC_IMPUTE = [
    "conso_5_usages_ep", "conso_5_usages_par_m2_ep",
    "conso_5 usages_ef", "conso_5 usages_par_m2_ef",
    "emission_ges_5_usages", "emission_ges_5_usages par_m2",
    "cout_total_5_usages", "cout_chauffage", "cout_ecs",
    "cout_refroidissement", "cout_eclairage", "cout_auxiliaires",
    "surface_habitable_logement", "nombre_niveau_logement",
    "conso_chauffage_ef",
]

# Calcul des médianes sur dpe_raw
medianes = {}
for col in NUMERIC_IMPUTE:
    q = f'SELECT MEDIAN(TRY_CAST(REPLACE(REPLACE("{col}", \',\', \'.\'), \' \', \'\') AS DOUBLE)) FROM dpe_raw WHERE "{col}" IS NOT NULL AND TRIM("{col}") != \'\''
    val = con.execute(q).fetchone()[0]
    medianes[col] = val if val is not None else 0
    print(f"  Médiane {col:<45} : {medianes[col]}")

# Construction du bloc SQL pour les numériques
median_expr = ",\n        ".join([
    f'COALESCE(TRY_CAST(REPLACE(REPLACE("{col}", \',\', \'.\'), \' \', \'\') AS DOUBLE), {medianes[col]}) AS "{col}"'
    for col in NUMERIC_IMPUTE
])

con.execute("DROP TABLE IF EXISTS dpe_imputed")

con.execute(f"""
    CREATE TABLE dpe_imputed AS
    SELECT
        -- Clés & dates
        numero_dpe,
        date_etablissement_dpe,
        date_fin_validite_dpe,
        date_derniere_modification_dpe,

        -- Géoloc BAN : fallback sur colonnes brutes si NULL
        COALESCE(NULLIF(TRIM(code_insee_ban),       ''), 'NR') AS code_insee_ban,
        COALESCE(NULLIF(TRIM(code_departement_ban), ''), 'NR') AS code_departement_ban,
        COALESCE(NULLIF(TRIM(code_region_ban),      ''), 'NR') AS code_region_ban,
        COALESCE(NULLIF(TRIM(nom_commune_ban),  ''), NULLIF(TRIM(nom_commune_brut),  ''), 'NR') AS nom_commune_ban,
        COALESCE(NULLIF(TRIM(code_postal_ban),  ''), NULLIF(TRIM(code_postal_brut),  ''), 'NR') AS code_postal_ban,
        COALESCE(TRY_CAST(coordonnee_cartographique_x_ban AS DOUBLE), -999) AS coordonnee_cartographique_x_ban,
        COALESCE(TRY_CAST(coordonnee_cartographique_y_ban AS DOUBLE), -999) AS coordonnee_cartographique_y_ban,
        COALESCE(TRY_CAST(score_ban AS DOUBLE), 0.0) AS score_ban,
        COALESCE(adresse_brut, '')          AS adresse_brut,
        COALESCE(nom_commune_brut, '')      AS nom_commune_brut,
        code_postal_brut,

        -- Bilan DPE
        UPPER(TRIM(etiquette_dpe)) AS etiquette_dpe,
        UPPER(TRIM(etiquette_ges)) AS etiquette_ges,

        -- Numériques : médiane
        {median_expr},

        -- Catégoriels : 'NR'
        COALESCE(NULLIF(TRIM(zone_climatique),  ''), 'NR') AS zone_climatique,
        COALESCE(NULLIF(TRIM(classe_altitude),  ''), 'NR') AS classe_altitude,
        COALESCE(NULLIF(TRIM(qualite_isolation_murs), ''), 'NR') AS qualite_isolation_murs,
        COALESCE(NULLIF(TRIM(type_generateur_chauffage_principal), ''), 'NR') AS type_generateur_chauffage_principal,
        COALESCE(NULLIF(TRIM(typologie_logement), ''), 'NR') AS typologie_logement,

        -- Colonnes sans NULL
        periode_construction,
        type_batiment,
        type_energie_principale_chauffage,
        qualite_isolation_enveloppe,
        qualite_isolation_menuiseries,
        ubat_w_par_m2_k,

        -- Valeurs par défaut explicites
        COALESCE(TRY_CAST(indicateur_confort_ete AS INTEGER), 0) AS indicateur_confort_ete,
        COALESCE(NULLIF(TRIM(isolation_toiture), ''), 'Inconnu') AS isolation_toiture

    FROM dpe_raw
""")

n = con.execute("SELECT COUNT(*) FROM dpe_imputed").fetchone()[0]
print(f"\n  dpe_imputed créée : {n:,} lignes")

# ── Vérification : zéro NULL ──────────────────────────────────────────────────
print("\n  Vérification nulls résiduels :")
all_cols = [row[0] for row in con.execute("DESCRIBE dpe_imputed").fetchall()]
has_null = False
for col in all_cols:
    n_null = con.execute(f'SELECT COUNT(*) FROM dpe_imputed WHERE "{col}" IS NULL').fetchone()[0]
    if n_null > 0:
        print(f" {col:<45} : {n_null:,} NULL restants")
        has_null = True
if not has_null:
    print("Aucun NULL — table prête pour l'analyse")

#-------------------------------------------------------------------------------   
# ── 5. Contrôle Qualité (QC) ──────────────────────────────────────────────────
#-------------------------------------------------------------------------------   

total = con.execute("SELECT COUNT(*) FROM dpe_imputed").fetchone()[0]
qc_log = []

print("=" * 65)
print("RAPPORT QUALITÉ DPE")
print("=" * 65)

# ── 5a. Unicité de la clé primaire numero_dpe ────────────────────────────────
n_null_id = con.execute("SELECT COUNT(*) FROM dpe_imputed WHERE numero_dpe IS NULL").fetchone()[0]
n_dup_id  = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT numero_dpe, COUNT(*) AS n
        FROM dpe_imputed
        WHERE numero_dpe IS NOT NULL
        GROUP BY numero_dpe HAVING n > 1
    )
""").fetchone()[0]
print(f"\n[CLÉ PRIMAIRE]")
print(f"  numero_dpe NULL     : {n_null_id:,}")
print(f"  numero_dpe dupliqué : {n_dup_id:,} valeurs distinctes en doublon")
if n_null_id: qc_log.append({"check": "null_numero_dpe",  "count": n_null_id})
if n_dup_id:  qc_log.append({"check": "dup_numero_dpe",   "count": n_dup_id})

# ── 5b. Étiquettes DPE / GES valides ─────────────────────────────────────────
print(f"\n[ÉTIQUETTES]")
for col in ["etiquette_dpe", "etiquette_ges"]:
    n_invalid = con.execute(f"""
        SELECT COUNT(*) FROM dpe_imputed
        WHERE "{col}" IS NOT NULL
          AND UPPER(TRIM("{col}")) NOT IN ('A','B','C','D','E','F','G')
    """).fetchone()[0]
    print(f"  {col:<30} : {n_invalid:,} valeurs invalides")
    if n_invalid: qc_log.append({"check": f"invalid_{col}", "count": n_invalid})

# Distribution étiquettes
print("\n  Distribution etiquette_dpe :")
dist = con.execute("""
    SELECT UPPER(TRIM(etiquette_dpe)) AS etiq, COUNT(*) AS n
    FROM dpe_imputed WHERE etiquette_dpe IS NOT NULL
    GROUP BY etiq ORDER BY etiq
""").fetchdf()
print(dist.to_string(index=False))


# ── 5d. Cohérence des dates ───────────────────────────────────────────────────
print(f"\n[COHÉRENCE DATES]")
n_date_bad = con.execute("""
    SELECT COUNT(*) FROM dpe_imputed
    WHERE date_fin_validite_dpe IS NOT NULL
      AND date_etablissement_dpe IS NOT NULL
      AND date_fin_validite_dpe < date_etablissement_dpe
""").fetchone()[0]
print(f"  date_fin < date_etablissement : {n_date_bad:,}")
if n_date_bad: qc_log.append({"check": "date_fin_avant_etablissement", "count": n_date_bad})

n_expires = con.execute("SELECT COUNT(*) FROM dpe_imputed WHERE DATE(date_fin_validite_dpe) < CURRENT_DATE").fetchone()[0]
print(f"  DPE expirés (date_fin < aujourd'hui) : {n_expires:,}  [info, pas une erreur]")

# ── 5e. Surfaces aberrantes ───────────────────────────────────────────────────
print(f"\n[SURFACES]")
n_surf_zero = con.execute("SELECT COUNT(*) FROM dpe_imputed WHERE surface_habitable_logement <= 0").fetchone()[0]
n_surf_huge = con.execute("SELECT COUNT(*) FROM dpe_imputed WHERE surface_habitable_logement > 2000").fetchone()[0]
print(f"  surface_habitable_logement <= 0    : {n_surf_zero:,}")
print(f"  surface_habitable_logement > 2000  : {n_surf_huge:,}")
if n_surf_zero: qc_log.append({"check": "surface_<=0", "count": n_surf_zero})
if n_surf_huge: qc_log.append({"check": "surface_>2000", "count": n_surf_huge})

# ── 5f. Consommation EP négative ──────────────────────────────────────────────
print(f"\n[CONSOMMATION]")
n_neg_conso = con.execute("SELECT COUNT(*) FROM dpe_imputed WHERE conso_5_usages_ep < 0").fetchone()[0]
print(f"  conso_5_usages_ep < 0             : {n_neg_conso:,}")
if n_neg_conso: qc_log.append({"check": "conso_ep_negative", "count": n_neg_conso})


# ── 5h. Score BAN hors [0, 1] ─────────────────────────────────────────────────
print(f"\n[GÉOCODAGE BAN]")
n_ban = con.execute("SELECT COUNT(*) FROM dpe_imputed WHERE score_ban < 0 OR score_ban > 1").fetchone()[0]
print(f"  score_ban hors [0,1]              : {n_ban:,}")
if n_ban: qc_log.append({"check": "score_ban_hors_range", "count": n_ban})


# ── Résumé QC ─────────────────────────────────────────────────────────────────
print(f"\n{'=' * 65}")
print(f"⚠️  Total anomalies détectées : {sum(x['count'] for x in qc_log):,} dans {len(qc_log)} catégories")
for item in sorted(qc_log, key=lambda x: -x['count']):
    print(f"  {item['check']:<50} : {item['count']:,}")

#-------------------------------------------------------------------------------   
# ── Dédoublonnage ─────────────────────────────────────────────────────────────
#-------------------------------------------------------------------------------   


con.execute("DROP TABLE IF EXISTS dpe_dedup")

con.execute("""
    CREATE TABLE dpe_final AS
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY numero_dpe
                   ORDER BY date_derniere_modification_dpe DESC NULLS LAST
               ) AS _rn
        FROM dpe_imputed
        WHERE numero_dpe IS NOT NULL
    )
    WHERE _rn = 1
""")

con.execute("ALTER TABLE dpe_final DROP COLUMN _rn")

n_imputed = con.execute("SELECT COUNT(*) FROM dpe_imputed").fetchone()[0]
n_final   = con.execute("SELECT COUNT(*) FROM dpe_final").fetchone()[0]
print(f"  dpe_final créée : {n_final:,} lignes ({n_imputed - n_final:,} doublons supprimés)")


DuckDB connecté.
  Table 'dpe_raw' supprimée (si existait)
  Table 'dpe_imputed' supprimée (si existait)
  Table 'dpe_clean' supprimée (si existait)
  Table 'dpe_dedup' supprimée (si existait)
  Table 'dpe_final' supprimée (si existait)
  Colonnes conservées : 47 / 47


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


  Lignes chargées : 120,000
  Colonnes en mémoire : 47
  Lignes totales : 120,000

  Colonne                                             NULL       %
  -----------------------------------------------------------------
  numero_dpe                                             0    0.0 %
  date_etablissement_dpe                                 0    0.0 %
  date_fin_validite_dpe                                  0    0.0 %
  date_derniere_modification_dpe                         0    0.0 %
  code_insee_ban                                    12,452   10.4 %  ℹ️
  code_departement_ban                              13,599   11.3 %  ℹ️
  code_region_ban                                   13,599   11.3 %  ℹ️
  coordonnee_cartographique_x_ban                   12,452   10.4 %  ℹ️
  coordonnee_cartographique_y_ban                   12,452   10.4 %  ℹ️
  score_ban                                         12,452   10.4 %  ℹ️
  nom_commune_ban                                   12,452   10.4 %  ℹ️
  cod